# 1. Introduction

`dsfb-bank` is the executable empirical companion for the DSFB theorem banks. This notebook rebuilds the crate from scratch in the current Colab session, runs the theorem-bank artifact from scratch, validates the new timestamped output directory, generates the paper figures from those fresh CSVs only, saves PNG exports, and packages the fresh artifact into a zip archive.

The workflow below is intentionally strict: no cached build artifacts are trusted, no prior output folder is reused, and missing files raise hard failures.

# 2. Environment Setup

This section locates the DSFB workspace root, refreshes the Colab clone to `origin/main` when `/content/dsfb` already exists, installs Python plotting dependencies if needed, records the notebook session start time, and captures the pre-run `output-dsfb-bank/` directory inventory so the later fresh-run check can prove that the chosen output directory was created in this session.

In [ ]:
import datetime as dt
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy", "pandas", "matplotlib"], check=True)
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

from IPython.display import Markdown, display

REPO_URL = "https://github.com/infinityabundance/dsfb.git"


def now_utc() -> str:
    return dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")


def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        cargo_toml = candidate / "Cargo.toml"
        if cargo_toml.exists() and "[workspace]" in cargo_toml.read_text():
            return candidate
    return None


def refresh_colab_clone(clone_root: Path) -> None:
    subprocess.run(["git", "fetch", "origin"], check=True, cwd=clone_root)
    subprocess.run(["git", "switch", "main"], check=True, cwd=clone_root)
    subprocess.run(["git", "pull", "--ff-only", "origin", "main"], check=True, cwd=clone_root)


SESSION_UTC = now_utc()
clone_root = Path("/content/dsfb")
repo_root = find_repo_root(Path.cwd())
if repo_root is None:
    if not clone_root.exists():
        subprocess.run(["git", "clone", REPO_URL, str(clone_root)], check=True)
    else:
        refresh_colab_clone(clone_root)
    repo_root = clone_root
elif repo_root.resolve() == clone_root.resolve():
    refresh_colab_clone(clone_root)

os.chdir(repo_root)
output_root = repo_root / "output-dsfb-bank"
preexisting_run_dirs = set()
if output_root.exists():
    preexisting_run_dirs = {path.resolve() for path in output_root.iterdir() if path.is_dir()}

repo_head = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=repo_root, text=True, capture_output=True, check=True).stdout.strip()

display(Markdown(f"Repository root: `{repo_root}`"))
display(Markdown(f"Repository commit: `{repo_head}`"))
display(Markdown(f"Notebook session UTC start: `{SESSION_UTC}`"))
display(Markdown(f"Pre-run output directories discovered: `{len(preexisting_run_dirs)}`"))

# 3. Fresh Rust Installation

This section installs Rust in the current session if needed and then verifies `cargo` and `rustc` before the clean rebuild.

In [ ]:
env = os.environ.copy()
cargo_bin = Path.home() / ".cargo" / "bin"
if shutil.which("cargo") is None or shutil.which("rustc") is None:
    subprocess.run("curl https://sh.rustup.rs -sSf | sh -s -- -y", shell=True, check=True)
    env["PATH"] = f"{cargo_bin}:{env['PATH']}"
else:
    env["PATH"] = f"{cargo_bin}:{env['PATH']}"

subprocess.run(["cargo", "--version"], check=True, cwd=repo_root, env=env)
subprocess.run(["rustc", "--version"], check=True, cwd=repo_root, env=env)

# 4. Fresh Build From Scratch

The notebook now performs an explicit clean build. The clean step is mandatory and the build step must succeed in the current session before any outputs are trusted.

In [ ]:
subprocess.run(["cargo", "clean", "-p", "dsfb-bank"], check=True, cwd=repo_root, env=env)
subprocess.run(["cargo", "build", "-p", "dsfb-bank"], check=True, cwd=repo_root, env=env)
build_completed_utc = now_utc()
display(Markdown(f"Fresh clean build completed at `{build_completed_utc}`."))

# 5. Fresh Theorem-Bank Run

This section runs `cargo run -p dsfb-bank -- --all` in the current session, captures the printed fresh output directory, and refuses to continue if the binary does not emit a valid run path.

In [ ]:
run_started_utc = now_utc()
run_proc = subprocess.run(
    ["cargo", "run", "-p", "dsfb-bank", "--", "--all"],
    cwd=repo_root,
    env=env,
    text=True,
    capture_output=True,
    check=True,
)

print(run_proc.stdout)
if run_proc.stderr.strip():
    print(run_proc.stderr)

stdout_lines = [line.strip() for line in run_proc.stdout.splitlines() if line.strip()]
assert stdout_lines, "dsfb-bank did not print a run directory on stdout"
run_dir = Path(stdout_lines[-1]).resolve()
assert run_dir.exists(), f"Fresh run directory does not exist: {run_dir}"
run_completed_utc = now_utc()
display(Markdown(f"Fresh run directory from current execution: `{run_dir}`"))

# 6. Locate Fresh Output Directory

The notebook now proves that the selected output directory was created by the current execution by comparing the pre-run and post-run `output-dsfb-bank/` inventories. Only that fresh directory is used for all remaining analysis.

In [ ]:
assert output_root.exists(), f"Expected output root to exist after the run: {output_root}"
post_run_dirs = {path.resolve() for path in output_root.iterdir() if path.is_dir()}
created_run_dirs = sorted(post_run_dirs - preexisting_run_dirs)

assert created_run_dirs, "No new timestamped output directory appeared in the current session"
assert run_dir in created_run_dirs, f"Printed run directory {run_dir} was not newly created in this session"
newest_created_run_dir = max(created_run_dirs, key=lambda path: path.name)
assert newest_created_run_dir == run_dir, (
    f"Newest created run directory {newest_created_run_dir} does not match the current run directory {run_dir}"
)

display(Markdown(f"Newest current-session run directory: `{newest_created_run_dir}`"))

# 7. Validate Output Inventory

This section fails loudly unless the current-session run produced the expected theorem-bank CSV inventory, `component_summary.csv`, and manifest case-class counts.


In [ ]:
manifest_path = run_dir / "manifest.json"
run_summary_path = run_dir / "run_summary.md"
logs_path = run_dir / "logs.txt"
component_summary_path = run_dir / "component_summary.csv"

assert manifest_path.exists(), f"Missing manifest.json: {manifest_path}"
assert run_summary_path.exists(), f"Missing run_summary.md: {run_summary_path}"
assert logs_path.exists(), f"Missing logs.txt: {logs_path}"
assert component_summary_path.exists(), f"Missing component_summary.csv: {component_summary_path}"

required_components = ["core", "dsfb", "dscd", "tmtr", "add", "srd", "hret"]
required_component_counts = {
    "core": 11,
    "dsfb": 20,
    "dscd": 20,
    "tmtr": 20,
    "add": 20,
    "srd": 20,
    "hret": 20,
}

component_csv_paths = {}
for component, expected_count in required_component_counts.items():
    component_dir = run_dir / component
    assert component_dir.exists(), f"Missing component directory: {component_dir}"
    csv_paths = sorted(component_dir.glob("*.csv"))
    component_csv_paths[component] = csv_paths
    assert len(csv_paths) == expected_count, (
        f"Expected {expected_count} theorem CSVs in {component_dir}, found {len(csv_paths)}"
    )

realizations_dir = run_dir / "realizations"
required_realization_paths = [
    realizations_dir / "dsfb_realizations.csv",
    realizations_dir / "dscd_realizations.csv",
    realizations_dir / "tmtr_realizations.csv",
    realizations_dir / "add_realizations.csv",
    realizations_dir / "srd_realizations.csv",
    realizations_dir / "hret_realizations.csv",
    realizations_dir / "all_realizations.csv",
]
for path in required_realization_paths:
    assert path.exists(), f"Missing realization CSV: {path}"

manifest = json.loads(manifest_path.read_text())
assert len(manifest["theorem_demos_run"]) == 131, "Expected 131 theorem demos in manifest.json"
case_class_counts = manifest.get("case_class_counts")
assert isinstance(case_class_counts, dict), "manifest.json is missing case_class_counts"
global_case_class_counts = case_class_counts.get("global")
by_component_case_class_counts = case_class_counts.get("by_component")
assert isinstance(global_case_class_counts, dict), "manifest.json is missing case_class_counts.global"
assert isinstance(by_component_case_class_counts, dict), "manifest.json is missing case_class_counts.by_component"
for key in ["passing", "boundary", "violating"]:
    assert key in global_case_class_counts, f"manifest.json is missing global case_class_counts[{key!r}]"
for component in required_components:
    assert component in by_component_case_class_counts, f"manifest.json is missing case_class_counts for {component}"
    for key in ["passing", "boundary", "violating"]:
        assert key in by_component_case_class_counts[component], (
            f"manifest.json is missing case_class_counts.by_component[{component!r}][{key!r}]"
        )

display(Markdown("Fresh output inventory validation passed."))


# 8. Load Fresh CSV Outputs

This section loads only the fresh CSVs from the current-session run directory, validates the hardened schemas, and cross-checks `component_summary.csv` and `manifest.json` against the actual emitted theorem rows.


In [ ]:
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.18

PALETTE = {
    "ink": "#1d3557",
    "river": "#457b9d",
    "mint": "#2a9d8f",
    "gold": "#e9c46a",
    "sand": "#f4a261",
    "alert": "#e76f51",
    "violet": "#6d597a",
    "forest": "#386641",
}
CASE_CLASS_COLORS = {
    "passing": PALETTE["forest"],
    "boundary": PALETTE["gold"],
    "violating": PALETTE["alert"],
}
component_order = ["core", "dsfb", "dscd", "tmtr", "add", "srd", "hret"]
bank_order = ["dsfb", "dscd", "tmtr", "add", "srd", "hret"]
case_class_order = ["passing", "boundary", "violating"]
required_common_columns = [
    "theorem_id",
    "theorem_name",
    "component",
    "case_id",
    "case_class",
    "assumption_satisfied",
    "expected_outcome",
    "observed_outcome",
    "pass",
    "notes",
]
required_columns_by_component = {
    "core": ["time_step", "signal_value", "observation_value", "reconstructed_state", "residual_value", "trust_value", "regime_label", "anomaly_flag"],
    "dsfb": ["injective_flag", "observation_id", "structural_state_id", "reconstructed_state_id", "residual_value", "exact_recovery_flag", "time_step", "signal_value", "observation_value", "reconstructed_state"],
    "dscd": ["graph_id", "node_count", "edge_count", "longest_path", "reachability_count", "acyclic_flag", "attempted_edge_addition_flag", "cycle_created_flag", "reduction_edge_count", "repaired_edge_count"],
    "tmtr": ["orbit_id", "iteration", "trust_value", "residual_value", "fixed_point_flag", "stabilization_iteration", "trust_gap", "trust_increase_attempt_flag", "trust_gap_satisfied_flag", "monotonicity_satisfied_flag"],
    "add": ["signal_id", "time_step", "signal_value", "residual_value", "first_difference", "second_difference", "threshold", "detector_output", "anomaly_magnitude"],
    "srd": ["trajectory_id", "time_step", "state_id", "fine_regime", "coarse_regime", "transition_flag", "coarse_transition_flag", "regime_valid_flag"],
    "hret": ["trace_id", "event_index", "trace_length", "prefix_length", "suffix_length", "observation_code", "reconstruction_success", "replayability_flag", "injective_observation_flag"],
}
component_summary_required_columns = ["component", "theorem_count", "cases", "pass", "fail", "boundary"]
component_summary_optional_columns = ["violating", "passing", "assumption_satisfied_count", "assumption_violated_count"]


def ensure_columns(df: pd.DataFrame, required_columns: list[str], path: Path) -> None:
    missing = [column for column in required_columns if column not in df.columns]
    assert not missing, f"{path} is missing required columns: {missing}"


def coerce_boolean_columns(df: pd.DataFrame) -> pd.DataFrame:
    for column in df.columns:
        if column == "pass" or column == "assumption_satisfied" or column.endswith("_flag") or column in {"reconstruction_success", "replayability_flag", "injective_observation_flag"}:
            df[column] = df[column].astype(str).str.lower().map({"true": True, "false": False})
    return df


def get_csv(component: str, prefix: str) -> tuple[Path, pd.DataFrame]:
    for path in component_csv_paths[component]:
        if path.name.startswith(prefix):
            return path, component_frames[component][path.name].copy()
    raise KeyError(f"Missing CSV for component={component!r} prefix={prefix!r}")


def slugify(title: str) -> str:
    slug = "".join(character.lower() if character.isalnum() else "_" for character in title)
    while "__" in slug:
        slug = slug.replace("__", "_")
    return slug.strip("_")


def category_codes(series: pd.Series) -> tuple[pd.Series, dict[str, int]]:
    categories = list(pd.Index(series.astype(str)).unique())
    mapping = {value: index for index, value in enumerate(categories)}
    return series.astype(str).map(mapping), mapping


figure_registry = []


def register_figure(figure_number: int, title: str, source_paths: list[Path], theorem_families: str, fig) -> None:
    figure_registry.append(
        {
            "figure_number": figure_number,
            "title": title,
            "source_csvs": sorted({str(path.relative_to(run_dir)) for path in source_paths}),
            "theorem_families": theorem_families,
            "figure": fig,
        }
    )


component_frames = {}
all_theorem_frames = []
for component, paths in component_csv_paths.items():
    frames = {}
    for path in paths:
        df = pd.read_csv(path)
        df = coerce_boolean_columns(df)
        ensure_columns(df, required_common_columns, path)
        ensure_columns(df, required_columns_by_component[component], path)
        invalid_case_classes = sorted(set(df["case_class"].astype(str)) - set(case_class_order))
        assert not invalid_case_classes, f"{path} contains invalid case_class values: {invalid_case_classes}"
        df["source_csv"] = str(path.relative_to(run_dir))
        frames[path.name] = df
        all_theorem_frames.append(df.copy())
    component_frames[component] = frames

all_theorem_rows = pd.concat(all_theorem_frames, ignore_index=True)
theorem_index = (
    all_theorem_rows.groupby(["component", "source_csv", "theorem_id", "theorem_name"], as_index=False)
    .agg(
        case_count=("case_id", "count"),
        pass_count=("pass", lambda series: int(series.fillna(False).sum())),
        fail_count=("pass", lambda series: int((series.fillna(False) == False).sum())),
    )
)
theorem_index["file_name"] = theorem_index["source_csv"].map(lambda value: Path(value).name)
assert theorem_index.shape[0] == 131, f"Expected 131 theorem CSVs, found {theorem_index.shape[0]}"

component_summary_df = pd.read_csv(component_summary_path)
ensure_columns(component_summary_df, component_summary_required_columns, component_summary_path)
for column in component_summary_required_columns[1:] + [column for column in component_summary_optional_columns if column in component_summary_df.columns]:
    component_summary_df[column] = component_summary_df[column].fillna(0).astype(int)
component_summary_df["component"] = pd.Categorical(component_summary_df["component"], categories=component_order, ordered=True)
component_summary_df = component_summary_df.sort_values("component").reset_index(drop=True)
assert component_summary_df.shape[0] == len(component_order), (
    f"Expected {len(component_order)} component_summary.csv rows, found {component_summary_df.shape[0]}"
)

base_components = pd.DataFrame({"component": component_order})
derived_component_metrics = all_theorem_rows.groupby("component", as_index=False).agg(
    **{
        "cases": ("case_id", "count"),
        "pass": ("pass", lambda series: int(series.fillna(False).sum())),
        "fail": ("pass", lambda series: int((series.fillna(False) == False).sum())),
        "boundary": ("case_class", lambda series: int((series.astype(str) == "boundary").sum())),
        "violating": ("case_class", lambda series: int((series.astype(str) == "violating").sum())),
        "passing": ("case_class", lambda series: int((series.astype(str) == "passing").sum())),
        "assumption_satisfied_count": ("assumption_satisfied", lambda series: int(series.fillna(False).sum())),
        "assumption_violated_count": ("assumption_satisfied", lambda series: int((series.fillna(False) == False).sum())),
    }
)
derived_component_summary = (
    base_components
    .merge(
        theorem_index.groupby("component", as_index=False)["theorem_id"].nunique().rename(columns={"theorem_id": "theorem_count"}),
        on="component",
        how="left",
    )
    .merge(derived_component_metrics, on="component", how="left")
    .fillna(0)
)
for column in ["theorem_count", "cases", "pass", "fail", "boundary", "violating", "passing", "assumption_satisfied_count", "assumption_violated_count"]:
    derived_component_summary[column] = derived_component_summary[column].astype(int)
comparison_columns = component_summary_required_columns[1:] + [column for column in component_summary_optional_columns if column in component_summary_df.columns]
component_summary_compare = component_summary_df[["component", *comparison_columns]].copy()
component_summary_compare["component"] = component_summary_compare["component"].astype(str)
derived_component_summary_compare = derived_component_summary[["component", *comparison_columns]].copy()
component_summary_validation = component_summary_compare.merge(
    derived_component_summary_compare,
    on="component",
    suffixes=("_file", "_derived"),
)
for column in comparison_columns:
    assert (component_summary_validation[f"{column}_file"] == component_summary_validation[f"{column}_derived"]).all(), (
        f"component_summary.csv does not match fresh theorem rows for column {column}"
    )

manifest_global_case_class_counts = {
    key: int(value)
    for key, value in manifest["case_class_counts"]["global"].items()
}
derived_global_case_class_counts = {
    case_class: int((all_theorem_rows["case_class"].astype(str) == case_class).sum())
    for case_class in case_class_order
}
assert manifest_global_case_class_counts == derived_global_case_class_counts, (
    f"manifest.json global case_class counts do not match fresh theorem rows: {manifest_global_case_class_counts} != {derived_global_case_class_counts}"
)
for component in component_order:
    manifest_component_counts = {
        key: int(value)
        for key, value in manifest["case_class_counts"]["by_component"][component].items()
    }
    derived_component_counts = {
        case_class: int(
            ((all_theorem_rows["component"] == component) & (all_theorem_rows["case_class"].astype(str) == case_class)).sum()
        )
        for case_class in case_class_order
    }
    assert manifest_component_counts == derived_component_counts, (
        f"manifest.json case_class counts for {component} do not match fresh theorem rows: {manifest_component_counts} != {derived_component_counts}"
    )

realization_frames = {}
for path in required_realization_paths:
    df = pd.read_csv(path)
    df["source_csv"] = str(path.relative_to(run_dir))
    realization_frames[path.name] = df

figure_output_dir = run_dir / "figures"
display(Markdown(f"Fresh theorem rows loaded: `{len(all_theorem_rows)}`"))
display(Markdown(f"Validated `component_summary.csv` and manifest case-class counts against the fresh theorem rows in `{run_dir}`."))


# 9. Generate Coverage Figures

This figure visualizes theorem-bank empirical coverage using all fresh theorem CSVs from the current run. Every heatmap cell counts witness cases for a `(theorem_id, component)` pair.


In [ ]:
coverage_index = theorem_index[theorem_index["component"].isin(bank_order)].copy()
coverage_theorem_counts = coverage_index.groupby("component")["theorem_id"].nunique().to_dict()
for component in bank_order:
    assert coverage_theorem_counts.get(component, 0) == 20, (
        f"Coverage heatmap expected 20 theorem IDs for {component}, found {coverage_theorem_counts.get(component, 0)}"
    )
coverage_heatmap = (
    coverage_index.pivot_table(index="theorem_id", columns="component", values="case_count", aggfunc="sum", fill_value=0)
    .reindex(columns=bank_order, fill_value=0)
    .sort_index()
)

fig, ax = plt.subplots(figsize=(8, 16))
image = ax.imshow(coverage_heatmap.values, aspect="auto", cmap="YlGnBu")
ax.set_xlabel("Component")
ax.set_ylabel("Theorem ID")
ax.set_xticks(range(len(coverage_heatmap.columns)))
ax.set_xticklabels(coverage_heatmap.columns)
ax.set_yticks(range(len(coverage_heatmap.index)))
ax.set_yticklabels(coverage_heatmap.index, fontsize=6)
ax.set_title("Figure 2 — Theorem Bank Coverage Heatmap")
for row_index in range(coverage_heatmap.shape[0]):
    for column_index in range(coverage_heatmap.shape[1]):
        value = int(coverage_heatmap.iat[row_index, column_index])
        if value > 0:
            ax.text(column_index, row_index, str(value), ha="center", va="center", fontsize=6, color="#0b1f2a")
fig.colorbar(image, ax=ax, label="Witness-case count")
fig.tight_layout()
register_figure(
    2,
    "Theorem Bank Coverage Heatmap",
    [run_dir / relative for relative in coverage_index["source_csv"].tolist()],
    "DSFB, DSCD, TMTR, ADD, SRD, HRET",
    fig,
)
plt.show()


# 10. Generate Behavior Figures

Figure 1 uses `core/grand_unification.csv` from the fresh run directory to show the aligned end-to-end DSFB stack trace on a shared time axis. Figure 3 uses all fresh TMTR theorem CSVs to show trust descent and stabilization without reusing prior outputs.


In [ ]:
path_core_hero, core_hero = get_csv("core", "grand_unification")
core_hero = core_hero.sort_values("time_step").copy()
regime_codes, regime_map = category_codes(core_hero["regime_label"])
anomaly_numeric = core_hero["anomaly_flag"].fillna(False).astype(int)

display(Markdown("Figure 1 source CSV: `core/grand_unification.csv` from the fresh run directory."))
fig, axes = plt.subplots(6, 1, figsize=(12, 15), sharex=True, gridspec_kw={"height_ratios": [1.8, 1.3, 1.1, 1.1, 1.0, 0.8]})
axes[0].plot(core_hero["time_step"], core_hero["signal_value"], marker="o", color=PALETTE["ink"], label="signal_value")
axes[0].plot(core_hero["time_step"], core_hero["observation_value"], marker="s", color=PALETTE["river"], label="observation_value")
axes[0].set_ylabel("Signal / observation")
axes[0].legend(loc="upper left")

axes[1].step(core_hero["time_step"], core_hero["reconstructed_state"], where="post", color=PALETTE["mint"], linewidth=2)
axes[1].scatter(core_hero["time_step"], core_hero["reconstructed_state"], color=PALETTE["mint"], s=45)
axes[1].set_ylabel("Reconstructed state")

residual_colors = [CASE_CLASS_COLORS["violating"] if flag else PALETTE["sand"] for flag in core_hero["anomaly_flag"].fillna(False)]
axes[2].bar(core_hero["time_step"], core_hero["residual_value"], color=residual_colors)
axes[2].set_ylabel("Residual")

axes[3].plot(core_hero["time_step"], core_hero["trust_value"], marker="o", color=PALETTE["violet"], linewidth=2)
axes[3].set_ylabel("Trust")

axes[4].step(core_hero["time_step"], regime_codes, where="post", color=PALETTE["forest"], linewidth=2)
axes[4].set_ylabel("Regime")
axes[4].set_yticks(list(regime_map.values()))
axes[4].set_yticklabels(list(regime_map.keys()))

axes[5].bar(core_hero["time_step"], anomaly_numeric, color=PALETTE["alert"], width=0.7)
axes[5].set_ylabel("Anomaly")
axes[5].set_yticks([0, 1])
axes[5].set_yticklabels(["no", "yes"])
axes[5].set_xlabel("Time step")

fig.suptitle("Figure 1 — Full DSFB Stack Timeline (End-to-End DSFB Stack Trace)")
fig.tight_layout()
register_figure(1, "Full DSFB Stack Timeline", [path_core_hero], "Core aligned DSFB stack", fig)
plt.show()

tmtr_paths = component_csv_paths["tmtr"]
tmtr_all = pd.concat([component_frames["tmtr"][path.name].copy() for path in tmtr_paths], ignore_index=True)
tmtr_unique = (
    tmtr_all.sort_values(["orbit_id", "iteration", "case_class", "source_csv"])
    .drop_duplicates(["orbit_id", "iteration", "trust_value", "next_trust_value", "case_class"])
)
display(Markdown("Figure 3 source CSVs: all fresh `tmtr/*.csv` theorem outputs from the current run."))
fig, ax = plt.subplots(figsize=(10, 6))
for orbit_id, frame in tmtr_unique.groupby("orbit_id"):
    frame = frame.sort_values("iteration")
    orbit_case_class = frame["case_class"].iloc[0]
    line_style = {"passing": "-", "boundary": ":", "violating": "--"}[orbit_case_class]
    color = CASE_CLASS_COLORS[orbit_case_class]
    ax.plot(frame["iteration"], frame["trust_value"], marker="o", linestyle=line_style, color=color, label=f"{orbit_id} ({orbit_case_class})")
    last_row = frame.iloc[-1]
    ax.annotate(
        f"stab={int(last_row['stabilization_iteration'])}",
        xy=(last_row["iteration"], last_row["trust_value"]),
        xytext=(6, 4),
        textcoords="offset points",
        fontsize=8,
        color=color,
    )
ax.set_xlabel("Iteration")
ax.set_ylabel("Trust value")
ax.set_title("Figure 3 — TMTR Trust Convergence")
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
register_figure(3, "TMTR Trust Convergence", tmtr_paths, "TMTR", fig)
plt.show()

path_dsfb_07, dsfb_07 = get_csv("dsfb", "07_")
fig, ax = plt.subplots(figsize=(8, 5))
bar_colors = dsfb_07["case_class"].map(CASE_CLASS_COLORS)
ax.bar(dsfb_07["case_id"], dsfb_07["residual_value"], color=bar_colors)
ax.set_xlabel("Case ID")
ax.set_ylabel("Residual value")
ax.set_title("Figure 7 — DSFB Residual Collapse")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
register_figure(7, "DSFB Residual Collapse", [path_dsfb_07], "DSFB", fig)
plt.show()

path_add_14, add_14 = get_csv("add", "14_")
fig, ax = plt.subplots(figsize=(10, 5))
for case_class, frame in add_14.groupby("case_class"):
    frame = frame.sort_values("time_step")
    ax.plot(frame["time_step"], frame["residual_value"], marker="o", color=CASE_CLASS_COLORS[case_class], label=f"{case_class} residual")
    ax.hlines(frame["threshold"].iloc[0], frame["time_step"].min(), frame["time_step"].max(), linestyle="--", color=CASE_CLASS_COLORS[case_class], alpha=0.6)
    triggered = frame[frame["detector_output"] != "none"]
    ax.scatter(triggered["time_step"], triggered["residual_value"], color=CASE_CLASS_COLORS[case_class], s=80)
ax.set_xlabel("Time step")
ax.set_ylabel("Residual / threshold")
ax.set_title("Figure 10 — ADD Residual Threshold Detection")
ax.legend()
fig.tight_layout()
register_figure(10, "ADD Residual Threshold Detection", [path_add_14], "ADD", fig)
plt.show()

path_add_03, add_03 = get_csv("add", "03_")
path_add_09, add_09 = get_csv("add", "09_")
path_add_10, add_10 = get_csv("add", "10_")
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
axes[0].plot(add_03["time_step"], add_03["signal_value"], marker="o", color=PALETTE["ink"], label="drift")
axes[0].plot(add_09["time_step"], add_09["signal_value"], marker="s", color=PALETTE["sand"], label="step")
axes[0].plot(add_10["time_step"], add_10["signal_value"], marker="^", color=PALETTE["alert"], label="spike")
axes[0].set_ylabel("Signal value")
axes[0].legend()
axes[1].plot(add_03["time_step"], add_03["first_difference"], marker="o", color=PALETTE["ink"], label="drift first diff")
axes[1].plot(add_09["time_step"], add_09["first_difference"], marker="s", color=PALETTE["sand"], label="step first diff")
axes[1].set_ylabel("First difference")
axes[1].legend()
axes[2].plot(add_10["time_step"], add_10["second_difference"], marker="^", color=PALETTE["alert"], label="spike second diff")
axes[2].set_xlabel("Time step")
axes[2].set_ylabel("Second difference")
axes[2].legend()
fig.suptitle("Figure 11 — ADD First/Second Difference Detection")
fig.tight_layout()
register_figure(11, "ADD First/Second Difference Detection", [path_add_03, path_add_09, path_add_10], "ADD", fig)
plt.show()

path_srd_03, srd_03 = get_csv("srd", "03_")
srd_transition = srd_03[srd_03["trajectory_id"] == "block_constant"].copy()
fine_codes, fine_map = category_codes(srd_transition["fine_regime"])
fig, ax = plt.subplots(figsize=(10, 4))
ax.step(srd_transition["time_step"], fine_codes, where="post", color=PALETTE["forest"], linewidth=2)
transition_points = srd_transition[srd_transition["transition_flag"] == True]
ax.scatter(transition_points["time_step"], fine_codes.loc[transition_points.index], color=PALETTE["alert"], s=70, label="transition")
ax.set_xlabel("Time step")
ax.set_ylabel("Fine regime")
ax.set_yticks(list(fine_map.values()))
ax.set_yticklabels(list(fine_map.keys()))
ax.set_title("Figure 12 — SRD Regime Transition Timeline")
ax.legend()
fig.tight_layout()
register_figure(12, "SRD Regime Transition Timeline", [path_srd_03], "SRD", fig)
plt.show()

path_srd_18, srd_18 = get_csv("srd", "18_")
srd_coarse = srd_18[srd_18["trajectory_id"] == "block_constant"].copy()
coarse_codes, coarse_map = category_codes(srd_coarse["coarse_regime"])
fig, ax = plt.subplots(figsize=(10, 4))
ax.step(srd_coarse["time_step"], coarse_codes, where="post", color=PALETTE["violet"], linewidth=2)
ax.set_xlabel("Time step")
ax.set_ylabel("Coarse regime")
ax.set_yticks(list(coarse_map.values()))
ax.set_yticklabels(list(coarse_map.keys()))
ax.set_title("Figure 13 — SRD Coarsened Regime Timeline")
fig.tight_layout()
register_figure(13, "SRD Coarsened Regime Timeline", [path_srd_18], "SRD", fig)
plt.show()

path_hret_09, hret_09 = get_csv("hret", "09_")
hret_success = (
    hret_09.groupby(["trace_id", "injective_observation_flag"], as_index=False)
    .agg(trace_length=("trace_length", "max"), reconstruction_success=("reconstruction_success", "mean"))
)
fig, ax = plt.subplots(figsize=(8, 5))
for injective_flag, frame in hret_success.groupby("injective_observation_flag"):
    label = "injective observation" if injective_flag else "non-injective observation"
    color = PALETTE["forest"] if injective_flag else PALETTE["alert"]
    ax.scatter(frame["trace_length"], frame["reconstruction_success"], color=color, s=80, label=label)
ax.set_xlabel("Trace length")
ax.set_ylabel("Reconstruction success rate")
ax.set_ylim(-0.05, 1.05)
ax.set_title("Figure 14 — HRET Trace Reconstruction Success")
ax.legend()
fig.tight_layout()
register_figure(14, "HRET Trace Reconstruction Success", [path_hret_09], "HRET", fig)
plt.show()


# 11. Generate Structural Figures

Figure 4 uses one actual row from `core/grand_unification.csv` to render the DSFB structural inference pipeline as a deterministic flow diagram. Figure 5 uses fresh TMTR CSVs with emitted `residual_value` and `trust_value` columns to show joint residual-trust convergence; the remaining structural figures reuse fresh DSFB and DSCD CSVs only.


In [ ]:
flow_candidates = core_hero.assign(anomaly_rank=core_hero["anomaly_flag"].fillna(False).astype(int))
flow_row = flow_candidates.sort_values(["anomaly_rank", "residual_value", "time_step"], ascending=[False, False, True]).iloc[0]
flow_source_relative = path_core_hero.relative_to(run_dir)
display(Markdown(
    f"Figure 4 source CSV: `{flow_source_relative}`; selected row `time_step={int(flow_row['time_step'])}` with `residual_value={float(flow_row['residual_value']):.3f}` and `anomaly_flag={bool(flow_row['anomaly_flag'])}`. This renders one actual deterministic inference path from raw signal to anomaly decision."
))

fig, ax = plt.subplots(figsize=(8, 12))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
flow_nodes = [
    ("Signal", f"value = {float(flow_row['signal_value']):.3f}", PALETTE["ink"]),
    ("Observation", f"value = {float(flow_row['observation_value']):.3f}", PALETTE["river"]),
    ("Reconstruction", f"state = {int(flow_row['reconstructed_state'])}", PALETTE["mint"]),
    ("Residual", f"value = {float(flow_row['residual_value']):.3f}", PALETTE["sand"]),
    ("Trust", f"value = {float(flow_row['trust_value']):.3f}", PALETTE["violet"]),
    ("Regime", f"label = {flow_row['regime_label']}", PALETTE["forest"]),
    ("Anomaly", f"flag = {'yes' if bool(flow_row['anomaly_flag']) else 'no'}", PALETTE["alert"]),
]
y_positions = np.linspace(0.92, 0.10, len(flow_nodes))
for index, (title, value_text, color) in enumerate(flow_nodes):
    y = y_positions[index]
    ax.text(
        0.5,
        y,
        f"{title}\n{value_text}",
        ha="center",
        va="center",
        fontsize=12,
        bbox=dict(boxstyle="round,pad=0.55", facecolor=color, edgecolor=color, alpha=0.18, linewidth=2.2),
    )
    if index < len(flow_nodes) - 1:
        ax.annotate(
            "",
            xy=(0.5, y_positions[index + 1] + 0.055),
            xytext=(0.5, y - 0.055),
            arrowprops=dict(arrowstyle="->", linewidth=2.0, color=PALETTE["ink"], shrinkA=0, shrinkB=0),
        )
fig.suptitle("Figure 4 — DSFB Structural Inference Flow")
fig.tight_layout()
register_figure(4, "DSFB Structural Inference Flow", [path_core_hero], "Core aligned DSFB stack", fig)
plt.show()

tmtr_phase = (
    tmtr_all.sort_values(["orbit_id", "iteration", "case_class", "source_csv"])
    .drop_duplicates(["orbit_id", "iteration", "residual_value", "trust_value", "case_class"])
    .copy()
)
required_phase_columns = ["orbit_id", "iteration", "trust_value", "residual_value"]
missing_phase_columns = [column for column in required_phase_columns if column not in tmtr_phase.columns]
assert not missing_phase_columns, f"TMTR phase plot is missing required columns: {missing_phase_columns}"
display(Markdown(
    "Figure 5 source CSVs: all fresh `tmtr/*.csv` theorem outputs from the current run. Each orbit is sorted by `iteration` and plotted in residual-versus-trust space to show whether the deterministic updates move toward a stable low-residual region."
))

fig, ax = plt.subplots(figsize=(9, 7))
ax.axvspan(0.0, 1.0, color=PALETTE["gold"], alpha=0.08)
ax.axhspan(0.0, 1.0, color=PALETTE["mint"], alpha=0.06)
for orbit_id, frame in tmtr_phase.groupby("orbit_id"):
    frame = frame.sort_values("iteration")
    orbit_case_class = frame["case_class"].iloc[0]
    color = CASE_CLASS_COLORS[orbit_case_class]
    line_style = {"passing": "-", "boundary": ":", "violating": "--"}[orbit_case_class]
    ax.plot(
        frame["residual_value"],
        frame["trust_value"],
        marker="o",
        linestyle=line_style,
        linewidth=2,
        color=color,
        label=f"{orbit_id} ({orbit_case_class})",
    )
    for idx in range(len(frame) - 1):
        start = frame.iloc[idx]
        end = frame.iloc[idx + 1]
        ax.annotate(
            "",
            xy=(end["residual_value"], end["trust_value"]),
            xytext=(start["residual_value"], start["trust_value"]),
            arrowprops=dict(arrowstyle="->", color=color, alpha=0.5, linewidth=1.2),
        )
    final_row = frame.iloc[-1]
    ax.annotate(
        orbit_id,
        xy=(final_row["residual_value"], final_row["trust_value"]),
        xytext=(6, 4),
        textcoords="offset points",
        fontsize=8,
        color=color,
    )
ax.set_xlabel("Residual value")
ax.set_ylabel("Trust value")
ax.set_title("Figure 5 — DSFB–TMTR Convergence Phase Plot")
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
register_figure(5, "DSFB TMTR Convergence Phase Plot", tmtr_paths, "TMTR", fig)
plt.show()

path_dsfb_02, dsfb_02 = get_csv("dsfb", "02_")
recoverability = (
    dsfb_02.assign(injective_label=np.where(dsfb_02["injective_flag"], "injective", "non_injective"))
    .groupby("injective_label", as_index=False)["equivalence_class_size"]
    .mean()
)
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(recoverability["injective_label"], recoverability["equivalence_class_size"], color=[PALETTE["forest"], PALETTE["alert"]])
ax.set_xlabel("Forward-map class")
ax.set_ylabel("Mean inverse-image size")
ax.set_title("Figure 6 — DSFB Recoverability vs Ambiguity")
fig.tight_layout()
register_figure(6, "DSFB Recoverability vs Ambiguity", [path_dsfb_02], "DSFB", fig)
plt.show()

path_dscd_08, dscd_08 = get_csv("dscd", "08_")
dscd_08_unique = dscd_08[["graph_id", "node_count", "longest_path"]].drop_duplicates()
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(dscd_08_unique["node_count"], dscd_08_unique["longest_path"], color=PALETTE["river"], s=70)
x_values = sorted(dscd_08_unique["node_count"].unique())
ax.plot(x_values, [value - 1 for value in x_values], linestyle="--", color=PALETTE["alert"], label="|V| - 1")
ax.set_xlabel("Node count")
ax.set_ylabel("Longest path")
ax.set_title("Figure 8 — DSCD DAG Size vs Longest Path")
ax.legend()
fig.tight_layout()
register_figure(8, "DSCD DAG Size vs Longest Path", [path_dscd_08], "DSCD", fig)
plt.show()

path_dscd_13, dscd_13 = get_csv("dscd", "13_")
dscd_13_unique = dscd_13[["graph_id", "edge_count", "reduction_edge_count", "reachability_count"]].drop_duplicates()
indices = np.arange(len(dscd_13_unique))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(indices - 0.15, dscd_13_unique["edge_count"], width=0.3, color=PALETTE["river"], label="original")
axes[0].bar(indices + 0.15, dscd_13_unique["reduction_edge_count"], width=0.3, color=PALETTE["ink"], label="reduced")
axes[0].set_xticks(indices)
axes[0].set_xticklabels(dscd_13_unique["graph_id"], rotation=30)
axes[0].set_ylabel("Edge count")
axes[0].set_title("Reduction")
axes[0].legend()
axes[1].bar(dscd_13_unique["graph_id"], dscd_13_unique["reachability_count"], color=PALETTE["gold"])
axes[1].set_ylabel("Reachability count")
axes[1].set_title("Reachability preserved")
axes[1].tick_params(axis="x", rotation=30)
fig.suptitle("Figure 9 — DSCD Reduction vs Reachability Preservation")
fig.tight_layout()
register_figure(9, "DSCD Reduction vs Reachability Preservation", [path_dscd_13], "DSCD", fig)
plt.show()


# 12. Save Figure PNGs

All notebook-generated figures are exported as individual PNG files into the fresh run directory under `figures/`.


In [ ]:
assert figure_registry, "No figures were registered for export"
assert not figure_output_dir.exists(), f"Refusing to reuse an existing figures directory: {figure_output_dir}"
figure_output_dir.mkdir(parents=True, exist_ok=False)

for item in sorted(figure_registry, key=lambda value: value["figure_number"]):
    file_name = f"figure_{item['figure_number']:02d}_{slugify(item['title'])}.png"
    png_path = figure_output_dir / file_name
    item["figure"].savefig(png_path, dpi=220, bbox_inches="tight")
    item["png_relative_path"] = str(png_path.relative_to(run_dir))
    item["png_path"] = str(png_path.resolve())
    plt.close(item["figure"])

saved_figure_pngs = sorted(figure_output_dir.glob("*.png"))
assert len(saved_figure_pngs) == len(figure_registry), "Figure PNG export count does not match the registered figure count"
display(Markdown(f"Saved `{len(saved_figure_pngs)}` figure PNG files to `{figure_output_dir.resolve()}`."))


# 13. Create Download Zip

The zip archive is an additional convenience artifact. The original CSV files and PNG files remain in the fresh run directory after archive creation.


In [ ]:
zip_path = run_dir / f"dsfb-bank-{run_dir.name}.zip"
assert not zip_path.exists(), f"Refusing to overwrite an existing zip archive: {zip_path}"

zip_members = []
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(run_dir.rglob("*.csv")):
        archive.write(path, arcname=str(path.relative_to(run_dir)))
        zip_members.append(str(path.relative_to(run_dir)))
    for path in saved_figure_pngs:
        archive.write(path, arcname=str(path.relative_to(run_dir)))
        zip_members.append(str(path.relative_to(run_dir)))
    for path in [manifest_path, run_summary_path, logs_path]:
        if path.exists():
            archive.write(path, arcname=str(path.relative_to(run_dir)))
            zip_members.append(str(path.relative_to(run_dir)))

assert zip_path.exists(), f"Zip archive was not created: {zip_path}"
with zipfile.ZipFile(zip_path, "r") as archive:
    archived_names = set(archive.namelist())
required_zip_members = {str(manifest_path.relative_to(run_dir)), str(run_summary_path.relative_to(run_dir))}
required_zip_members.update(str(path.relative_to(run_dir)) for path in run_dir.rglob("*.csv"))
required_zip_members.update(str(path.relative_to(run_dir)) for path in saved_figure_pngs)
if logs_path.exists():
    required_zip_members.add(str(logs_path.relative_to(run_dir)))
missing_zip_members = sorted(required_zip_members - archived_names)
assert not missing_zip_members, f"Zip archive is missing required members: {missing_zip_members}"

display(Markdown(f"Zip archive created at `{zip_path.resolve()}`"))


# 14. Display Summary Tables

These tables summarize the fresh CSV inventory, the theorem-bank execution counts, the saved figure manifest, and the final downloadable archive for the current run only.


In [ ]:
csv_inventory = pd.DataFrame(
    [
        {"relative_path": str(path.relative_to(run_dir)), "bytes": path.stat().st_size}
        for path in sorted(run_dir.rglob("*.csv"))
    ]
)

theorem_counts = component_summary_df[["component", "theorem_count"]].copy()
theorem_counts["component"] = theorem_counts["component"].astype(str)
pass_fail_counts = component_summary_df[["component", "cases", "pass", "fail"]].copy()
pass_fail_counts["component"] = pass_fail_counts["component"].astype(str)
case_class_counts = component_summary_df[["component", "passing", "boundary", "violating"]].copy()
case_class_counts["component"] = case_class_counts["component"].astype(str)
component_summary_preview = component_summary_df.copy()
component_summary_preview["component"] = component_summary_preview["component"].astype(str)

figure_manifest_df = pd.DataFrame(
    [
        {
            "figure_number": item["figure_number"],
            "figure_title": item["title"],
            "source_csv_paths": ", ".join(item["source_csvs"]),
            "theorem_families_supported": item["theorem_families"],
            "saved_png_path": item["png_path"],
        }
        for item in sorted(figure_registry, key=lambda value: value["figure_number"])
    ]
)

zip_summary = pd.DataFrame(
    [{"zip_path": str(zip_path.resolve()), "zip_bytes": zip_path.stat().st_size}]
)

display(Markdown("#### CSV inventory table"))
display(csv_inventory)

display(Markdown("#### Theorem counts by component"))
display(theorem_counts)

display(Markdown("#### Pass/fail counts by component"))
display(pass_fail_counts)

display(Markdown("#### Case-class counts by component"))
display(case_class_counts)

display(Markdown("#### component_summary.csv preview"))
display(component_summary_preview)

display(Markdown("#### Figure manifest table"))
display(figure_manifest_df)

display(Markdown("#### Zip archive path and file size"))
display(zip_summary)


# 15. Reproducibility Confirmation

The statements below summarize what this notebook verified about the current-session build, run, figures, and archive.


In [ ]:
repro_lines = [
    f"- The crate was rebuilt in this notebook session after an explicit `cargo clean -p dsfb-bank` step. Build completion time: `{build_completed_utc}`.",
    f"- The theorem bank was executed in this notebook session via `cargo run -p dsfb-bank -- --all`. Run window: `{run_started_utc}` to `{run_completed_utc}`.",
    f"- The output directory `{run_dir}` was created in this notebook session and was absent from the pre-run directory inventory.",
    f"- `component_summary.csv` and `manifest.json` case-class counts were validated against the fresh theorem rows loaded from `{run_dir}` in this session.",
    f"- All figures were generated only from fresh CSV outputs loaded from `{run_dir}` in this session.",
    f"- The PNG exports were saved individually to `{figure_output_dir.resolve()}` in this session.",
    f"- The zip archive `{zip_path.resolve()}` was created from those fresh CSV files, figure PNGs, and run metadata in this session.",
]
display(Markdown("\n".join(repro_lines)))
